# Taller 2 · Construimos un agente paso a paso

**Fundación Monge · Runbook guiado en Python**

En este taller trasladaremos a Python el alter ego conversacional que construimos previamente en n8n. La idea es aprender observando: ejecutamos una versión pequeña, vemos qué hace, agregamos una capacidad y volvemos a probar.

El estudiante solo necesita personalizar algunos valores del alter ego y escribir mensajes de prueba propios. El notebook usa Python, el OpenAI SDK, SQLite y Gradio; evitamos frameworks complejos.

## Mapa del recorrido

1. Preparar el ambiente y configurar la API.
2. Personalizar la identidad del alter ego.
3. Ver un agente en vivo sin memoria ni herramientas.
4. Agregar memoria temporal y observar la diferencia.
5. Cambiar a memoria persistente con SQLite.
6. Agregar una herramienta para guardar recuerdos.
7. Agregar una herramienta para leer recuerdos.
8. Agregar la hora local y probarla.
9. Agregar el clima actual mediante Open-Meteo.
10. Abrir el chat gráfico del agente final.

Cada sección incluye una explicación, una celda ejecutable, un resultado esperado y un reto breve.


## 0. Preparación del ambiente virtual

Un ambiente virtual es una carpeta aislada para las librerías de este proyecto.

Desde la Terminal, ubicado en esta carpeta, ejecuta:

    python3 -m venv .venv
    source .venv/bin/activate
    python -m pip install --upgrade pip
    python -m pip install -r requirements.txt
    python -m ipykernel install --user --name taller-2-agente --display-name "Python (Taller 2 Agente)"

Después abre este notebook y selecciona el kernel Python (Taller 2 Agente).

In [30]:
import sys
from pathlib import Path

print('Python en uso:', sys.executable)
print('Versión:', sys.version.split()[0])
print('Carpeta:', Path.cwd())

Python en uso: /Users/rmonge/Documents/Py Experiments/Project 1/agents/.venv/bin/python
Versión: 3.12.13
Carpeta: /Users/rmonge/Documents/GitHub/Taller de IA parte 2


**Resultado esperado:** la ruta de Python debería contener .venv.

**Reto:** identifica qué parte de la ruta indica que estás dentro del ambiente virtual.

## 1. Instalamos dependencias

Usaremos:

- openai para llamar al modelo;
- python-dotenv para cargar .env;
- gradio para crear el chat gráfico;
- sqlite3, que ya viene incluido con Python, para la memoria local.

In [31]:
%pip install -q -U openai python-dotenv gradio ipykernel
print('Dependencias listas.')

/Users/rmonge/Documents/Py Experiments/Project 1/agents/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
Dependencias listas.


**Resultado esperado:** aparecerá Dependencias listas.

**Reto:** explica por qué sqlite3 no aparece en el comando de instalación.

## 2. Cargamos .env y creamos el cliente

El archivo .env debe estar junto al notebook y contener:

    OPENAI_API_KEY=tu_clave

El archivo no se carga automáticamente. load_dotenv lo incorpora al ambiente del proceso. Usamos override=True para evitar que una variable antigua de VS Code o de la Terminal gane por accidente.

In [32]:
import os
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd()
ENV_FILE = ROOT / '.env'
load_dotenv(ENV_FILE, override=True)

api_key = os.getenv('OPENAI_API_KEY')
MODEL = os.getenv('OPENAI_MODEL', 'gpt-5-mini')

print('Archivo .env:', ENV_FILE)
print('API key encontrada:', bool(api_key))
print('Modelo:', MODEL)

if not api_key:
    raise RuntimeError('No se encontró OPENAI_API_KEY en .env')

client = OpenAI(api_key=api_key)
print('Cliente OpenAI listo.')

Archivo .env: /Users/rmonge/Documents/GitHub/Taller de IA parte 2/.env
API key encontrada: True
Modelo: gpt-5-mini
Cliente OpenAI listo.


**Resultado esperado:** API key encontrada: True y Cliente OpenAI listo. La clave nunca se imprime.

Si aparece un error 401, la API rechazó la clave: reemplázala por una válida y reinicia el kernel. Si aparece un error de archivo, confirma que el notebook se ejecuta desde la carpeta del taller.

**Reto:** cambia OPENAI_MODEL en .env por otro modelo de texto disponible en tu cuenta.

## 3. Personalizamos el alter ego

Cada estudiante cambia los valores de este diccionario. Mantén las mismas claves para que el resto del notebook funcione.

In [33]:
ALTER_EGO = {
    'nombre': 'Luna',
    'descripcion': 'una estudiante curiosa que disfruta aprender y explicar ideas',
    'personalidad': ['amable', 'creativa', 'entusiasta'],
    'gustos': ['la música', 'la tecnología', 'los retos'],
    'objetivo': 'ayudar a otras personas a aprender sin hacerles la tarea',
    'estilo': 'responde en español, con claridad, ejemplos cortos y un tono cercano'
}

MEMORY_ID = 'mi-alter-ego'

print(f"Alter ego: {ALTER_EGO['nombre']}")
print(f"Gustos: {', '.join(ALTER_EGO['gustos'])}")
print(f'Memoria: {MEMORY_ID}')

Alter ego: Luna
Gustos: la música, la tecnología, los retos
Memoria: mi-alter-ego


**Resultado esperado:** aparecerán el nombre, los gustos y el identificador de memoria.

**Reto:** cambia nombre, personalidad, gustos y estilo. Si varias personas comparten computador, usa un MEMORY_ID distinto.

## 4. Construimos el prompt de sistema

El prompt de sistema define identidad, tono y reglas. También le explica al modelo en qué situaciones debe solicitar nuestras tools.

In [34]:
def crear_prompt_sistema(ego):
    return f'''
Eres {ego['nombre']}, {ego['descripcion']}.
Tu personalidad incluye: {', '.join(ego['personalidad'])}.
Te interesan: {', '.join(ego['gustos'])}.
Tu objetivo es: {ego['objetivo']}.

Reglas:
- {ego['estilo']}.
- Eres un personaje creado para este taller; no afirmes ser una persona real.
- Si no sabes algo, dilo con honestidad.
- Si la persona comparte un nombre, gusto o preferencia estable, usa guardar_recuerdo.
- Si preguntan qué recuerdas, usa leer_recuerdos antes de responder.
- Si preguntan la hora actual, usa obtener_hora_local.
- Si preguntan por el clima, usa obtener_clima_actual e indica la ciudad consultada.
- Nunca inventes recuerdos ni afirmes que guardaste algo si la tool no se ejecutó.
'''.strip()

SYSTEM_PROMPT = crear_prompt_sistema(ALTER_EGO)
print(SYSTEM_PROMPT)

Eres Luna, una estudiante curiosa que disfruta aprender y explicar ideas.
Tu personalidad incluye: amable, creativa, entusiasta.
Te interesan: la música, la tecnología, los retos.
Tu objetivo es: ayudar a otras personas a aprender sin hacerles la tarea.

Reglas:
- responde en español, con claridad, ejemplos cortos y un tono cercano.
- Eres un personaje creado para este taller; no afirmes ser una persona real.
- Si no sabes algo, dilo con honestidad.
- Si la persona comparte un nombre, gusto o preferencia estable, usa guardar_recuerdo.
- Si preguntan qué recuerdas, usa leer_recuerdos antes de responder.
- Si preguntan la hora actual, usa obtener_hora_local.
- Si preguntan por el clima, usa obtener_clima_actual e indica la ciudad consultada.
- Nunca inventes recuerdos ni afirmes que guardaste algo si la tool no se ejecutó.


**Resultado esperado:** se imprimirán las instrucciones completas del agente.

**Reto:** agrega una regla de estilo que te represente.

## 5. Primera versión: un agente en vivo sin memoria ni tools

Comenzamos con el agente más pequeño posible. Recibe un mensaje, lo envía al modelo junto con el prompt de sistema y muestra la respuesta.

En esta versión el agente no guarda conversaciones, no conoce datos anteriores y no puede ejecutar funciones externas. Eso nos da una línea base para comparar las siguientes versiones.


In [ ]:

def responder_sin_memoria(mensaje):
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=mensaje,
        store=False
    )
    return response.output_text

print(responder_sin_memoria('Preséntate en una sola frase.'))


**Resultado esperado:** aparecerá una respuesta del alter ego.

**Reto:** cambia el mensaje de prueba para pedirle una recomendación relacionada con los gustos de tu alter ego. Ejecuta la celda otra vez.

**Observación:** si preguntas en otra celda por un dato mencionado antes, el agente no tiene forma de recordarlo.


In [ ]:

print(responder_sin_memoria('¿Cuál es mi nombre?'))


La respuesta probablemente indicará que todavía no conoce tu nombre. Este comportamiento es intencional: todavía no hemos construido memoria.


## 6. Memoria temporal: una lista de Python

Memoria significa conservar información útil de turnos anteriores para usarla después. Primero construiremos una memoria sencilla en una lista.

Esta memoria vive únicamente mientras el notebook está ejecutándose. Si reiniciamos el kernel o cerramos el programa, se pierde.


In [ ]:

historial_temporal = []

def responder_con_memoria_temporal(mensaje):
    historial_temporal.append({'role': 'user', 'content': mensaje})

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=historial_temporal,
        store=False
    )

    historial_temporal.append({
        'role': 'assistant',
        'content': response.output_text
    })
    return response.output_text

print(responder_con_memoria_temporal('Mi nombre es Alex y me gusta dibujar.'))
print()
print(responder_con_memoria_temporal('¿Cómo me llamo y qué me gusta hacer?'))


**Resultado esperado:** en el segundo turno el agente debería responder usando el nombre y el gusto mencionados en el primero.

**Reto:** cambia el nombre y el gusto del primer mensaje. Luego ejecuta la celda desde el inicio de esta sección.

**Idea clave:** esta memoria funciona, pero está mezclando la conversación completa en una lista de Python. Es útil para aprender, aunque todavía no es una memoria que sobreviva a un reinicio.


## 7. Memoria persistente con SQLite, todavía sin tools

Ahora guardaremos los mensajes en una base de datos local SQLite. SQLite viene incluido con Python y crea un archivo pequeño dentro del proyecto.

Separaremos dos tipos de información:

- messages: turnos de la conversación.
- facts: datos importantes que queremos recordar sobre la persona.

Todavía no usaremos tools. La función de Python guardará los mensajes directamente, para que primero entendamos la persistencia.


In [35]:

import sqlite3
from datetime import datetime
from pathlib import Path

# Si esta celda se ejecuta de forma aislada, usamos valores seguros por defecto.
ROOT = globals().get('ROOT', Path.cwd())
MEMORY_ID = globals().get('MEMORY_ID', 'mi-alter-ego')
DB_PATH = ROOT / 'alter_ego_memory.db'

class SQLiteMemory:
    def __init__(self, path):
        self.path = str(path)
        self.legacy_facts = False
        self._create_tables()
        self.legacy_facts = self._detect_legacy_facts()

    def _connect(self):
        return sqlite3.connect(self.path)

    def _detect_legacy_facts(self):
        with self._connect() as conn:
            columns = {row[1] for row in conn.execute('PRAGMA table_info(facts)')}
        return {'fact_key', 'fact_value'}.issubset(columns)

    def _create_tables(self):
        with self._connect() as conn:
            conn.execute('''
                CREATE TABLE IF NOT EXISTS messages (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    memory_id TEXT NOT NULL,
                    role TEXT NOT NULL,
                    content TEXT NOT NULL,
                    created_at TEXT NOT NULL
                )
            ''')
            conn.execute('''
                CREATE TABLE IF NOT EXISTS facts (
                    memory_id TEXT NOT NULL,
                    fact_key TEXT NOT NULL,
                    fact_value TEXT NOT NULL,
                    updated_at TEXT NOT NULL,
                    PRIMARY KEY (memory_id, fact_key)
                )
            ''')
            conn.commit()

    def save_message(self, role, content):
        with self._connect() as conn:
            conn.execute(
                'INSERT INTO messages (memory_id, role, content, created_at) VALUES (?, ?, ?, ?)',
                (MEMORY_ID, role, content, datetime.now().isoformat(timespec='seconds'))
            )
            conn.commit()

    def get_messages(self, limit=20):
        with self._connect() as conn:
            rows = conn.execute(
                'SELECT role, content FROM messages WHERE memory_id = ? '
                'ORDER BY id DESC LIMIT ?',
                (MEMORY_ID, limit)
            ).fetchall()
        return [{'role': role, 'content': content} for role, content in reversed(rows)]

    def save_fact(self, fact):
        with self._connect() as conn:
            timestamp = datetime.now().isoformat(timespec='microseconds')
            if self.legacy_facts:
                conn.execute(
                    'INSERT INTO facts (memory_id, fact_key, fact_value, updated_at) VALUES (?, ?, ?, ?)',
                    (MEMORY_ID, f'recuerdo_{timestamp.replace(":", "").replace(".", "")}', fact, timestamp)
                )
            else:
                conn.execute(
                    'INSERT INTO facts (memory_id, fact, created_at) VALUES (?, ?, ?)',
                    (MEMORY_ID, fact, timestamp)
                )
            conn.commit()

    def get_facts(self, limit=20):
        with self._connect() as conn:
            if self.legacy_facts:
                rows = conn.execute(
                    'SELECT fact_value FROM facts WHERE memory_id = ? '
                    'ORDER BY updated_at DESC LIMIT ?',
                    (MEMORY_ID, limit)
                ).fetchall()
            else:
                rows = conn.execute(
                    'SELECT fact FROM facts WHERE memory_id = ? '
                    'ORDER BY id DESC LIMIT ?',
                    (MEMORY_ID, limit)
                ).fetchall()
        return [row[0] for row in reversed(rows)]

    def clear(self):
        with self._connect() as conn:
            conn.execute('DELETE FROM messages WHERE memory_id = ?', (MEMORY_ID,))
            conn.execute('DELETE FROM facts WHERE memory_id = ?', (MEMORY_ID,))
            conn.commit()

memory = SQLiteMemory(DB_PATH)
print('Base de datos:', DB_PATH)
print('Mensajes guardados:', len(memory.get_messages()))
print('Recuerdos guardados:', len(memory.get_facts()))


Base de datos: /Users/rmonge/Documents/GitHub/Taller de IA parte 2/alter_ego_memory.db
Mensajes guardados: 10
Recuerdos guardados: 1


**Resultado esperado:** aparecerá la ruta del archivo alter_ego_memory.db y dos conteos.

Si ya habías ejecutado una versión anterior del taller, esta celda reconoce automáticamente la tabla facts antigua (fact_key y fact_value), así que no necesitas borrar tus datos.

**Reto:** ejecuta la celda dos veces. El archivo y las tablas se reutilizan porque SQLite es persistente.

**Importante:** una base de datos no hace que el modelo recuerde automáticamente. Nosotros todavía debemos leer los datos y colocarlos en el contexto que recibe el modelo.


In [ ]:

# Recuperamos variables previas si el estudiante ejecutó esta sección aislada.
MODEL = globals().get('MODEL', 'gpt-5-mini')
if 'SYSTEM_PROMPT' not in globals():
    if 'crear_prompt_sistema' in globals() and 'ALTER_EGO' in globals():
        SYSTEM_PROMPT = crear_prompt_sistema(ALTER_EGO)
    else:
        SYSTEM_PROMPT = 'Eres un alter ego conversacional amable. Responde en español y con honestidad.'
if 'client' not in globals():
    raise RuntimeError('Falta el cliente OpenAI. Ejecuta primero la sección 2: Cargamos .env y creamos el cliente.')

# Leemos los recuerdos guardados para convertirlos en texto comprensible
# para el modelo. Esta es la conexión entre SQLite y la conversación.
def construir_contexto_memoria():
    hechos = memory.get_facts()
    hechos_texto = '\n'.join(f'- {hecho}' for hecho in hechos)
    if not hechos_texto:
        hechos_texto = '(todavía no hay recuerdos guardados)'

    return (
        'Recuerdos persistentes sobre la persona:\n'
        + hechos_texto
        + '\n\nUsa estos recuerdos solo cuando sean relevantes.'
    )

def responder_con_memoria_sin_tools(mensaje):
    # Guardamos el mensaje antes de consultar la base de datos para que
    # el turno actual también forme parte del historial.
    memory.save_message('user', mensaje)

    input_items = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'system', 'content': construir_contexto_memoria()},
        *memory.get_messages(limit=20)
    ]

    response = client.responses.create(
        model=MODEL,
        input=input_items,
        store=False
    )
    respuesta = response.output_text
    # Guardamos también la respuesta del modelo y completamos el turno.
    memory.save_message('assistant', respuesta)
    return respuesta

memory.clear()
print(responder_con_memoria_sin_tools('Mi nombre es Alex y me gusta dibujar.'))
print()
print(responder_con_memoria_sin_tools('¿Qué recuerdas de esta conversación?'))
print()
print('Mensajes en SQLite:', len(memory.get_messages()))


**Resultado esperado:** el segundo turno puede usar la conversación guardada. Si reinicias el kernel y vuelves a crear SQLiteMemory con el mismo archivo, los mensajes siguen allí.

**Reto:** cambia MEMORY_ID por otro identificador y observa que ese alter ego empieza con otra memoria separada.


## 8. ¿Qué es una tool y por qué la necesitamos?

Una tool es una función que programamos en Python para que el modelo pueda pedir una acción concreta. El modelo decide cuándo necesita la función, pero Python es quien la ejecuta.

El flujo es:

1. El modelo recibe una descripción de las tools.
2. Si hace falta, devuelve una solicitud de función.
3. Python ejecuta la función.
4. Python devuelve el resultado al modelo.
5. El modelo redacta la respuesta final.

La primera tool será guardar_recuerdo. Así el modelo podrá decidir cuándo un dato merece quedar guardado en la tabla facts.


In [ ]:
import json

# Esta es la función real que Python ejecutará cuando el modelo solicite
# la tool guardar_recuerdo. El modelo no ejecuta Python directamente.
def guardar_recuerdo(recuerdo, detalle=''):
    # Convertimos los argumentos recibidos en un texto sencillo para guardar.
    texto = recuerdo.strip()
    if detalle.strip():
        texto = f'{texto} | {detalle.strip()}'
    memory.save_fact(texto)
    return {'ok': True, 'recuerdo_guardado': texto}

# TOOLS_GUARDAR es la descripción que enviamos al modelo.
# Aquí explicamos el nombre de la función, su propósito y sus parámetros.
# Esta descripción guía la decisión del modelo: no contiene la función Python.
TOOLS_GUARDAR = [
    {
        'type': 'function',
        'name': 'guardar_recuerdo',
        'description': 'Guarda un dato estable o una preferencia importante de la persona.',
        'parameters': {
            'type': 'object',
            'properties': {
                'recuerdo': {
                    'type': 'string',
                    'description': 'Dato breve que debe recordarse.'
                },
                'detalle': {
                    'type': 'string',
                    'description': 'Contexto adicional opcional.'
                }
            },
            'required': ['recuerdo', 'detalle'],
            'additionalProperties': False
        },
        'strict': True
    }
]

# Este diccionario conecta el nombre que el modelo devuelve con la función
# que Python debe ejecutar. Podemos agregar más tools incorporando nuevas
# funciones al diccionario y nuevas descripciones a la lista de tools.
TOOL_FUNCTIONS_GUARDAR = {
    'guardar_recuerdo': guardar_recuerdo
}

print('Tool disponible:', TOOLS_GUARDAR[0]['name'])


**Resultado esperado:** aparecerá guardar_recuerdo.

Con strict=True, todas las propiedades declaradas deben aparecer en required. Por eso recuerdo y detalle son obligatorios para el esquema, aunque detalle pueda llegar vacío.

**Reto:** cambia la descripción para indicar qué tipo de dato debería evitar guardar la tool. La descripción funciona como una instrucción para el modelo.


In [ ]:

# El modelo devuelve el nombre de la tool y sus argumentos en un objeto.
# Esta función traduce esa solicitud a una llamada real de Python.
def ejecutar_tool(item, tool_functions):
    # Leemos el nombre exacto que el modelo eligió.
    nombre = item.name
    # Los argumentos llegan como texto JSON; los convertimos a un diccionario.
    argumentos = json.loads(item.arguments or '{}')

    # Buscamos la función usando el mismo nombre que aparece en el esquema.
    # Si no existe, devolvemos un error controlado en vez de detener todo.
    if nombre not in tool_functions:
        return {'ok': False, 'error': f'Tool no disponible: {nombre}'}

    try:
        resultado = tool_functions[nombre](**argumentos)
        return {'ok': True, 'tool': nombre, 'resultado': resultado}
    except Exception as error:
        return {'ok': False, 'tool': nombre, 'error': str(error)}

# Esta función coordina la conversación completa cuando existen tools.
# Recibe tres cosas: el mensaje, las descripciones de tools y el diccionario
# que permite ejecutar esas tools en Python.
def responder_con_tools(mensaje, tools, tool_functions):
    # PRIMERA PARTE DE LA MEMORIA: guardamos el nuevo mensaje del usuario
    # en SQLite antes de llamar al modelo. Así este turno queda persistido.
    memory.save_message('user', mensaje)

    # SEGUNDA PARTE DE LA MEMORIA: reconstruimos el contexto que recibirá
    # el modelo. Incluimos el prompt, los recuerdos y los últimos mensajes.
    # La base de datos no habla con el modelo por sí sola; nosotros leemos
    # sus datos y los agregamos explícitamente a input_items.
    input_items = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'system', 'content': construir_contexto_memoria()},
        *memory.get_messages(limit=20)
    ]

    # Guardaremos aquí una explicación de las tools usadas en este turno.
    # Este registro sirve para aprender, depurar y mostrar actividad en la UI.
    tool_log = []

    # Puede ser necesario hacer varias rondas: una para pedir una tool y otra
    # para que el modelo lea el resultado. El límite evita ciclos infinitos.
    for _ in range(4):
        # TERCERA PARTE DE LAS TOOLS: enviamos sus descripciones al modelo.
        # El parámetro tools=tools permite que el modelo decida si necesita
        # una función. Python todavía no ejecuta nada en esta línea.
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_PROMPT,
            input=input_items,
            tools=tools,
            tool_choice='auto',
            store=False
        )

        # Revisamos si la respuesta contiene una solicitud de función.
        # Si no hay function_call, el modelo ya produjo una respuesta normal.
        function_calls = [
            item for item in response.output
            if getattr(item, 'type', None) == 'function_call'
        ]

        if not function_calls:
            # CUARTA PARTE DE LA MEMORIA: guardamos la respuesta final
            # para que pueda aparecer en los siguientes turnos.
            respuesta = response.output_text
            memory.save_message('assistant', respuesta)
            return respuesta, tool_log

        # Conservamos los objetos originales de la Responses API.
        # Esto permite que el siguiente llamado entienda qué tool solicitó.
        # No usamos model_dump(), porque puede agregar campos no aceptados
        # por la API, como status.
        input_items += response.output

        for item in function_calls:
            # QUINTA PARTE DE LAS TOOLS: ahora sí ejecutamos la función
            # Python asociada al nombre elegido por el modelo.
            resultado = ejecutar_tool(item, tool_functions)
            # Guardamos información visible sobre lo que ocurrió.
            tool_log.append({
                'nombre': item.name,
                'argumentos': json.loads(item.arguments or '{}'),
                'resultado': resultado
            })
            # SEXTA PARTE DE LAS TOOLS: devolvemos el resultado a la API.
            # call_id conecta este resultado con la solicitud original.
            # Después de recibirlo, el modelo redactará la respuesta final.
            input_items.append({
                'type': 'function_call_output',
                'call_id': item.call_id,
                'output': json.dumps(resultado, ensure_ascii=False)
            })

    raise RuntimeError('El agente superó el máximo de rondas de tools.')

memory.clear()
respuesta, tool_log = responder_con_tools(
    'Mi nombre es Alex y me gusta dibujar. Guarda esos datos.',
    TOOLS_GUARDAR,
    TOOL_FUNCTIONS_GUARDAR
)
print(respuesta)
print()
print('Registro de tools:', tool_log)
print('Recuerdos:', memory.get_facts())


**Resultado esperado:** se mostrará la respuesta del modelo, un registro de la tool ejecutada y el recuerdo en SQLite.

Observa la diferencia con la sección anterior: el agente ya no solo conversa. Puede pedir una acción de Python y usar el resultado para responder.

**Reto:** prueba un mensaje que no pida guardar nada. Compara el registro de tools.


## 9. Agregamos una tool para leer recuerdos

Guardar información sirve poco si el agente no puede consultarla. Agregaremos leer_recuerdos.

Esta tool no cambia la base de datos: solamente recupera información y la entrega al modelo. Separar tools de lectura y escritura hace que el comportamiento sea más claro.


In [ ]:

def leer_recuerdos():
    return {
        'ok': True,
        'recuerdos': memory.get_facts(limit=20)
    }

TOOL_LEER_RECUERDOS = {
    'type': 'function',
    'name': 'leer_recuerdos',
    'description': 'Consulta los datos que el agente ha guardado sobre la persona.',
    'parameters': {
        'type': 'object',
        'properties': {},
        'required': [],
        'additionalProperties': False
    },
    'strict': True
}

# Para agregar otra tool, anexamos su esquema a la lista anterior.
# También agregamos su función al diccionario que Python puede ejecutar.
TOOLS_MEMORIA = TOOLS_GUARDAR + [TOOL_LEER_RECUERDOS]
TOOL_FUNCTIONS_MEMORIA = {
    **TOOL_FUNCTIONS_GUARDAR,
    'leer_recuerdos': leer_recuerdos
}

respuesta, tool_log = responder_con_tools(
    '¿Qué recuerdas de mí? Usa la memoria guardada.',
    TOOLS_MEMORIA,
    TOOL_FUNCTIONS_MEMORIA
)
print(respuesta)
print()
print('Registro de tools:', tool_log)


**Resultado esperado:** el agente debería solicitar leer_recuerdos y mencionar datos guardados.

**Reto:** guarda otro dato con una nueva prueba y vuelve a preguntar qué recuerda.


## 10. Agregamos la hora local

Una tool también puede consultar información que cambia con el tiempo. La hora no debe inventarla el modelo: Python puede obtenerla de una fuente confiable.

Esta función usa la biblioteca estándar de Python y una zona horaria válida, por ejemplo America/Costa_Rica.


In [ ]:

from datetime import datetime
from zoneinfo import ZoneInfo

def obtener_hora_local(zona_horaria='America/Costa_Rica'):
    ahora = datetime.now(ZoneInfo(zona_horaria))
    return {
        'ok': True,
        'zona_horaria': zona_horaria,
        'hora': ahora.strftime('%H:%M:%S'),
        'fecha': ahora.strftime('%Y-%m-%d')
    }

TOOL_HORA = {
    'type': 'function',
    'name': 'obtener_hora_local',
    'description': 'Obtiene la fecha y hora actuales de una zona horaria.',
    'parameters': {
        'type': 'object',
        'properties': {
            'zona_horaria': {
                'type': 'string',
                'description': 'Zona IANA, por ejemplo America/Costa_Rica.'
            }
        },
        'required': ['zona_horaria'],
        'additionalProperties': False
    },
    'strict': True
}

# La tool de hora se suma a las tools de memoria sin reemplazarlas.
TOOLS_HORA = TOOLS_MEMORIA + [TOOL_HORA]
TOOL_FUNCTIONS_HORA = {
    **TOOL_FUNCTIONS_MEMORIA,
    'obtener_hora_local': obtener_hora_local
}

respuesta, tool_log = responder_con_tools(
    '¿Qué hora es ahora en Costa Rica?',
    TOOLS_HORA,
    TOOL_FUNCTIONS_HORA
)
print(respuesta)
print()
print('Registro de tools:', tool_log)


**Resultado esperado:** la respuesta debe incluir una hora actual y el registro debe mostrar obtener_hora_local.

**Reto:** prueba America/New_York o Europe/Madrid. Luego vuelve a Costa Rica.


## 11. Agregamos clima actual con Open-Meteo

El clima es una buena tool real porque combina dos pasos:

1. Buscar las coordenadas de una ciudad.
2. Consultar las condiciones actuales para esas coordenadas.

Usaremos Open-Meteo, un servicio externo gratuito para este ejercicio. No agregamos otra API key al notebook. Como toda API externa puede fallar, la función devuelve un resultado controlado cuando la ciudad no existe o el servicio no responde.


In [ ]:

from urllib.parse import urlencode
from urllib.request import Request, urlopen

WEATHER_CODES = {
    0: 'cielo despejado',
    1: 'principalmente despejado',
    2: 'parcialmente nublado',
    3: 'nublado',
    45: 'niebla',
    48: 'niebla engelante',
    51: 'llovizna ligera',
    61: 'lluvia ligera',
    63: 'lluvia moderada',
    65: 'lluvia fuerte',
    80: 'chubascos ligeros',
    81: 'chubascos moderados',
    82: 'chubascos fuertes',
    95: 'tormenta'
}

def consultar_json(url, params):
    query = urlencode(params)
    request = Request(f'{url}?{query}', headers={'User-Agent': 'taller-agentes-ia/1.0'})
    with urlopen(request, timeout=10) as response:
        return json.loads(response.read().decode('utf-8'))

def obtener_clima_actual(ciudad):
    try:
        lugares = consultar_json(
            'https://geocoding-api.open-meteo.com/v1/search',
            {'name': ciudad, 'count': 1, 'language': 'es', 'format': 'json'}
        ).get('results', [])

        if not lugares:
            return {'ok': False, 'error': f'No encontré la ciudad: {ciudad}'}

        lugar = lugares[0]
        clima = consultar_json(
            'https://api.open-meteo.com/v1/forecast',
            {
                'latitude': lugar['latitude'],
                'longitude': lugar['longitude'],
                'current': 'temperature_2m,relative_humidity_2m,apparent_temperature,weather_code,wind_speed_10m',
                'timezone': 'auto'
            }
        )
        actual = clima['current']
        return {
            'ok': True,
            'ciudad': lugar.get('name', ciudad),
            'pais': lugar.get('country', ''),
            'temperatura_c': actual.get('temperature_2m'),
            'sensacion_c': actual.get('apparent_temperature'),
            'humedad_pct': actual.get('relative_humidity_2m'),
            'viento_kmh': actual.get('wind_speed_10m'),
            'condicion': WEATHER_CODES.get(actual.get('weather_code'), 'condición no catalogada'),
            'hora_medicion': actual.get('time'),
            'zona_horaria': clima.get('timezone')
        }
    except Exception as error:
        return {'ok': False, 'error': f'No se pudo consultar el clima: {error}'}

TOOL_CLIMA = {
    'type': 'function',
    'name': 'obtener_clima_actual',
    'description': 'Consulta el clima actual de una ciudad usando Open-Meteo.',
    'parameters': {
        'type': 'object',
        'properties': {
            'ciudad': {
                'type': 'string',
                'description': 'Nombre de la ciudad; incluye el país si puede haber ambigüedad, por ejemplo San José, Costa Rica.'
            }
        },
        'required': ['ciudad'],
        'additionalProperties': False
    },
    'strict': True
}

# Finalmente agregamos clima a todas las capacidades anteriores.
TOOLS_FINAL = TOOLS_HORA + [TOOL_CLIMA]
TOOL_FUNCTIONS_FINAL = {
    **TOOL_FUNCTIONS_HORA,
    'obtener_clima_actual': obtener_clima_actual
}

resultado_clima = obtener_clima_actual('San José, Costa Rica')
print(json.dumps(resultado_clima, ensure_ascii=False, indent=2))


**Resultado esperado:** aparecerá un objeto con ciudad, país, temperatura, humedad y condición actual.

**Reto:** cambia San José, Costa Rica por tu ciudad. Si no hay conexión, revisa el resultado de error y continúa: el agente está preparado para informar el problema.


## 12. Agente final: todas las capacidades juntas

Ya tenemos las piezas:

- identidad y prompt de sistema;
- memoria persistente con SQLite;
- una tool para guardar recuerdos;
- una tool para leerlos;
- una tool para consultar la hora;
- una tool para consultar el clima.

El agente final usa exactamente el mismo ciclo de conversación que probamos antes. La interfaz gráfica solo será otra forma de enviar mensajes y mostrar resultados.


In [ ]:

TOOLS = TOOLS_FINAL
TOOL_FUNCTIONS = TOOL_FUNCTIONS_FINAL

def responder_agente_final(mensaje):
    return responder_con_tools(mensaje, TOOLS, TOOL_FUNCTIONS)
    
print(responder_agente_final('Dime qué recuerdas de mí y el clima actual en San José, Costa Rica.')[0])


**Resultado esperado:** el agente puede combinar memoria y clima en una sola respuesta. En el registro de tools se pueden observar las funciones que decidió usar.

**Reto:** cambia la ciudad y formula una pregunta que combine dos capacidades distintas.


## 13. Construimos la interfaz gráfica

Gradio convierte nuestras funciones de Python en una pantalla de chat. Añadiremos:

- conversación visible;
- caja de texto y botón Enviar;
- botón para borrar la memoria de este alter ego;
- panel de actividad para ver qué tools se ejecutaron;
- panel de recuerdos persistentes.

La interfaz ayuda a probar el agente como una aplicación real, pero la lógica importante sigue estando en las funciones que ya construimos.


In [ ]:

import html
import gradio as gr

def cargar_chat_inicial():
    return memory.get_messages(limit=20)

def responder_ui(mensaje, historial):
    mensaje = (mensaje or '').strip()
    if not mensaje:
        return historial, '', 'Escribe un mensaje para comenzar.', ''

    try:
        respuesta, tool_log = responder_agente_final(mensaje)
        nuevos_mensajes = memory.get_messages(limit=20)
        herramientas = (
            json.dumps(tool_log, ensure_ascii=False, indent=2)
            if tool_log else 'No se ejecutaron tools en este turno.'
        )
        estado = f'Listo · {len(tool_log)} tool(s) ejecutada(s)'
        return nuevos_mensajes, '', estado, herramientas
    except Exception as error:
        return historial, mensaje, f'Error: {error}', ''

def borrar_memoria_ui():
    memory.clear()
    return [], 'Memoria borrada para este alter ego.', 'No hay recuerdos guardados.'

CSS = '''
body { background: #f4f0ea; }
.gradio-container { max-width: 1100px !important; margin: auto; }
#titulo { color: #2e2924; }
#subtitulo { color: #6f655d; }
button.primary { background: #9a5c36 !important; border-color: #9a5c36 !important; }
'''

with gr.Blocks(title=f'{ALTER_EGO["nombre"]} · Agente IA', css=CSS) as demo:
    gr.Markdown(
        f'# {html.escape(ALTER_EGO["nombre"])}',
        elem_id='titulo'
    )
    gr.Markdown(
        'Alter ego conversacional · memoria SQLite · tools reales',
        elem_id='subtitulo'
    )

    with gr.Row():
        with gr.Column(scale=2):
            chat = gr.Chatbot(
                value=cargar_chat_inicial,
                type='messages',
                label='Conversación',
                height=520
            )
            mensaje = gr.Textbox(
                label='Tu mensaje',
                placeholder='Pregunta algo, comparte un dato o consulta el clima...',
                lines=2
            )
            with gr.Row():
                enviar = gr.Button('Enviar', variant='primary')
                limpiar = gr.Button('Borrar memoria')

        with gr.Column(scale=1):
            estado = gr.Markdown('Listo.')
            herramientas = gr.Code(
                label='Tools ejecutadas en el último turno',
                language='json',
                value='No hay actividad todavía.'
            )
            recuerdos = gr.JSON(
                label='Recuerdos guardados',
                value=memory.get_facts()
            )

    enviar.click(
        responder_ui,
        inputs=[mensaje, chat],
        outputs=[chat, mensaje, estado, herramientas],
        show_progress='full'
    )
    mensaje.submit(
        responder_ui,
        inputs=[mensaje, chat],
        outputs=[chat, mensaje, estado, herramientas],
        show_progress='full'
    )
    limpiar.click(
        borrar_memoria_ui,
        outputs=[chat, estado, recuerdos]
    )

print('Interfaz construida. Ejecuta la siguiente celda para abrirla.')


**Resultado esperado:** aparecerá Interfaz construida. Todavía no se abre el navegador.

**Reto:** cambia el texto de bienvenida, el placeholder o el color principal. No modifiques la lógica de las tools.


## 14. Abrimos el chat final

Ejecuta esta celda para abrir la aplicación en el navegador. Luego prueba, en orden:

1. Mi nombre es Alex y me gusta dibujar.
2. ¿Qué recuerdas de mí?
3. ¿Qué hora es ahora en Costa Rica?
4. ¿Cuál es el clima actual en San José, Costa Rica?
5. Pregunta algo que combine memoria y clima.

Observa el panel de tools después de cada turno. Esa evidencia permite explicar qué parte resolvió el modelo y qué parte ejecutó Python.


In [ ]:

demo.launch(
    inbrowser=True,
    prevent_thread_lock=True,
    show_error=True,
    share=False
)


## Cierre y siguientes mejoras

El agente final ya tiene una arquitectura completa para el taller:

mensaje del usuario → modelo → decisión de tool → función Python → resultado → respuesta → memoria SQLite → interfaz gráfica.

Ideas para continuar:

- agregar una tool de calculadora;
- añadir una tool para buscar una actividad o una noticia;
- mejorar la política para decidir qué datos guardar;
- separar la base de datos por estudiante;
- agregar pruebas automáticas para cada tool;
- diseñar una personalidad distinta para cada alter ego.

Antes de compartir el notebook, recuerda no incluir claves reales dentro del archivo ni subir el archivo .env al repositorio.
